# Processing Data from the ESA Project Results Repository via the CDSE openEO Federation

In this final guide, we demonstrate how to use the **CDSE openEO Federation** to reuse and process data published in the [**ESA Project Results Repository**](https://eoresults.esa.int/browser/#/external/eoresults.esa.int/stac?.language=en) through **EarthCODE**.

We leverage openEO’s [`load_stac`](https://dataspace.copernicus.eu/news/2024-6-3-openeo-introduces-loadstac) functionality to define a datacube that references and processes data hosted on a different openEO platform. This approach enables seamless, federated data access and processing without the need to manually move or duplicate datasets.


Before diving into the actual code, we first define a small helper function that will assist with visualizing the final results.

In [1]:
import rasterio
import matplotlib.pyplot as plt

def visualise_tif(path: str):
    with rasterio.open(path) as src:
        data = src.read(1)  # Read the first band
        plt.figure(figsize=(10, 10))
        plt.imshow(data, cmap='YlGn')
        plt.colorbar()
        plt.show()

## Querying the ESA Project Results Repository

We begin by exploring the ESA Project Results Repository (PRR) using the `pystac_client` library. The PRR is based on the STAC specification, which allows us to programmatically browse and discover available collections.

In this example, we focus on the *FCM-AGB-100m* collection. Using the STAC client, we retrieve the URL that defines this collection. This URL is a key input for the next steps, as it allows us to access and process the data through openEO.

In [2]:
from pystac_client import Client

In [3]:
catalog_url = "https://eoresults.esa.int/stac"
cat = Client.open(catalog_url)

In [4]:
collections = cat.collection_search(q='Biomass').collections_as_dicts()
collection_id = "FCM-AGB-100m"

for c in collections:
    if c['id'] == collection_id:
        collection = c
        collection_url = list(filter(lambda x: x['rel'] == 'self', c['links']))[0]['href']
    
print(f"\nFound collection: {collection['id']} - {collection['title']} with URL: {collection_url}")


Found collection: FCM-AGB-100m - ESA FCM 100 m European-wide Above Ground Biomass with URL: https://eoresults.esa.int/stac/collections/FCM-AGB-100m


## Connection with CDSE openEO Federation

Before we can actually process the *FCM-AGB-100m* collection, we need to authenticate with an available openEO backend. In this example, we will use the CDSE openEO federation, which provides seamless access to both datasets and processing resources across multiple federated openEO backends.

In [5]:
import openeo

In [6]:
connection = openeo.connect(url="openeofed.dataspace.copernicus.eu").authenticate_oidc()

Authenticated using refresh token.


## Processing the Data with `load_stac`

With the connection established, we can now use openEO’s `load_stac` function to access and process the data.

In this notebook, we demonstrate a simple use case: applying spatial and temporal filters to the STAC collection and downloading the resulting subset. This example highlights the basic usage of `load_stac` within an openEO workflow.

In practice, `load_stac` can be combined with additional openEO processes to build more advanced analyses, experiments, or end-to-end processing workflows.


In [7]:
dataset = connection.load_stac(
    url=collection_url, 
    spatial_extent= {
        "coordinates": [
          [
            [
              5.1001801731459295,
              51.3422096694828
            ],
            [
              5.1001801731459295,
              51.19555518611335
            ],
            [
              5.390030580796946,
              51.19555518611335
            ],
            [
              5.390030580796946,
              51.3422096694828
            ],
            [
              5.1001801731459295,
              51.3422096694828
            ]
          ]
        ],
        "type": "Polygon"
      },
    temporal_extent= ["2023-01-01", "2023-01-01"]
)

bands_from_stac_collection: consulting items for band metadata
bands_from_stac_collection: no band name source found


Finally we download and visualise the results by launching an openEO batch job.

In [8]:
path =  "./files/biomass_data.tiff"
job = dataset.execute_batch(
    path,
    title="CDSE Federation - ESA PRR Example", 
    description="This is an example on how to load and process data from the ESA Project Results Repository",
)

0:00:00 Job 'cdse-j-260217072958400cbf3578af140a42d0': send 'start'
0:00:16 Job 'cdse-j-260217072958400cbf3578af140a42d0': queued (progress 0%)
0:00:22 Job 'cdse-j-260217072958400cbf3578af140a42d0': queued (progress 0%)
0:00:31 Job 'cdse-j-260217072958400cbf3578af140a42d0': queued (progress 0%)
0:00:40 Job 'cdse-j-260217072958400cbf3578af140a42d0': queued (progress 0%)
0:00:50 Job 'cdse-j-260217072958400cbf3578af140a42d0': queued (progress 0%)
0:01:03 Job 'cdse-j-260217072958400cbf3578af140a42d0': queued (progress 0%)
0:01:18 Job 'cdse-j-260217072958400cbf3578af140a42d0': error (progress N/A)
Your batch job 'cdse-j-260217072958400cbf3578af140a42d0' failed. Error logs:
[{'id': '[1771313438426, 278280]', 'time': '2026-02-17T07:30:38.426Z', 'level': 'error', 'message': "OpenEO batch job failed: OpenEOApiException(status_code=500, code='Internal', message='load_stac: Error when constructing datacube from https://eoresults.esa.int/stac/collections/FCM-AGB-100m: An error occurred while calli

JobFailedException: Batch job 'cdse-j-260217072958400cbf3578af140a42d0' didn't finish successfully. Status: error (after 0:01:20).

In [ ]:
visualise_tif(path)